# Wedding planner — a coordinator and her specialists 💍

**DJEF Wedding** — a small Module 2 project: a **multi-agent team** (the lesson
`2_3` wiring) built from the "Coordinator Agent" spec — one **coordinator** that
does not do the work itself, delegating to **three specialists**, each hidden
behind an ordinary `@tool`, each backed by a *real* tool:

| specialist | its real tool | parameter mapping |
|---|---|---|
| **flights** | `search_flights` through a **local MCP server** (`mcp_flights_server.py`, stdio) | `Origin` |
| **venue** | `web_search` — **live web search** (Tavily) | `Destination`, `Guest_Count` |
| **playlist** | `run_sql` — **SQL** over a local SQLite database | `Genre` |

The coordinator's prompt *is* the Coordinator Agent ROLE spec: extract
`Origin`, `Destination`, `Guest_Count`, `Genre` → **validate** (ask when one is
missing) → **dispatch** the three sub-agents → **synthesize** one answer shaped
✈️ **Flights** / 🏛️ **Venue** / 🎵 **Playlist**.

The app keeps its **wedding sidebar untouched** — guests, wilaya, style, date,
budget, priorities — and *adds* the two new parameters underneath: **Origin**
(where guests fly in from) and **Playlist genre**. The Wilaya doubles as the
coordinator's `Destination`, the guest count as `Guest_Count`.

The notebook is the bench; `streamlit/app.py` is the small app — the UI split
into `design.py` / `components.py` / `runner.py` — built on the same service
module (`wedding_planner_service.py`). `.env` at the project root:
`GOOGLE_API_KEY` required, `TAVILY_API_KEY` for the venues. Same small, cheap
model as every notebook in this path: `gemini-3.1-flash-lite`.

> Flight rows are a **demo dataset** served by the MCP server; venue results are
> live web links — the honesty note travels with every answer.


## Setup

In [1]:
import importlib.util as _ilu
import subprocess as _sp
import sys as _sys

def _check_install(pkg: str, mod: str):
    if _ilu.find_spec(mod) is None:
        result = _sp.run([_sys.executable, "-m", "pip", "install", "-q", pkg],
                         capture_output=True, text=True)
        print(f"{pkg}: installed" if result.returncode == 0
              else f"{pkg}: install FAILED -> {result.stderr[-200:]}")
    else:
        print(f"{pkg}: already installed")

for _pkg, _mod in [
    ("python-dotenv", "dotenv"),
    ("langchain", "langchain"),
    ("langgraph", "langgraph"),
    ("google-genai", "google.genai"),
    ("tavily-python", "tavily"),
    ("mcp", "mcp"),
    ("langchain-mcp-adapters", "langchain_mcp_adapters"),
]:
    _check_install(_pkg, _mod)

print("Setup done.")


python-dotenv: already installed
langchain: already installed
langgraph: already installed
google-genai: already installed
tavily-python: already installed
mcp: already installed
langchain-mcp-adapters: already installed
Setup done.


In [2]:
# Make the service importable wherever Jupyter was started from.
import os
import sys
from pathlib import Path

for _base in (Path.cwd(), *Path.cwd().parents):
    for _folder in (
        _base,
        _base / "notebooks",
        _base / "project_wedding_planner" / "notebooks",
        _base / "Module 2 - Advanced Agent" / "project_wedding_planner" / "notebooks",
    ):
        if (_folder / "wedding_planner_service.py").is_file():
            if str(_folder) not in sys.path:
                sys.path.insert(0, str(_folder))
            break
    else:
        continue
    break

# Load the Gemini key from the nearest .env (project folder) — never inside the file.
try:
    import wedding_planner_service as wp
except ImportError as _exc:
    raise RuntimeError(
        "Wrong/broken packages in this kernel. In Jupyter: "
        "Kernel -> Change kernel -> Python (djef), then Run All Cells. "
        f"(python: {sys.executable} | {_exc})"
    ) from _exc

source = wp.load_keys()
print("Key source:", source)
print("GOOGLE_API_KEY:", "set" if os.getenv("GOOGLE_API_KEY") else "missing")
print("GEMINI_API_KEY:", "set" if os.getenv("GEMINI_API_KEY") else "missing")
print("TAVILY_API_KEY:", "set" if os.getenv("TAVILY_API_KEY") else "missing (venues only)")
print("Setup done.")


Key source: C:\Users\Kassouar Fatima\Downloads\LangChain\Module 2 - Advanced Agent\project_wedding_planner\.env
GOOGLE_API_KEY: set
GEMINI_API_KEY: set
TAVILY_API_KEY: set
Setup done.


In [3]:
# Same small, cheap Gemini model as every notebook in this path.
MODEL = wp.EVENT_MODEL
print("Model:", MODEL)


Model: google_genai:gemini-3.1-flash-lite


In [4]:
import sys
import asyncio

# Fixes for Windows issues in Jupyter notebooks (launching MCP stdio servers)
if sys.platform == "win32":
    # 1. Use the ProactorEventLoop policy, which supports subprocesses
    if not isinstance(asyncio.get_event_loop_policy(),
                      asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid the fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

print("Windows fixes applied:", sys.platform)


Windows fixes applied: win32


## 1. Flights — the MCP server

The Flights Agent's tool does **not** live in our process: it is exposed by a small
**MCP server** (`mcp_flights_server.py`) that speaks JSON-RPC over **stdio**.
`langchain-mcp-adapters` connects, runs `initialize` + `tools/list`, and hands us
back an ordinary LangChain tool — the agent never sees the protocol:

```python
tools = wp.flights_mcp_tools()          # spawn server -> tools/list
tools[0].name, tools[0].description
```

The rows are a **demo dataset** (typical routes towards Algeria) and every answer
repeats that note — nobody mistakes it for a live quote. The service gives each
async MCP tool a *sync twin*, because our agent — Streamlit and this notebook —
call tools synchronously.


In [5]:
import json

tools = wp.flights_mcp_tools()
print("discovered:", [t.name for t in tools])
print("about     :", tools[0].description.splitlines()[0])

data = json.loads(tools[0].invoke({"origin": "Paris"}))
print(f"\nDemo flights from {data['origin']}:")
for flight in data["flights"]:
    print(f"  {flight['airline']:20} {flight['flight']:9} {flight['route']:34}"
          f" {flight['depart']:>10} -> {flight['arrive']}")
print("note:", data["note"])


discovered: ['search_flights']
about     : Find flights departing from an origin city towards the event in Algeria.

Demo flights from Paris:
  Air Algérie          AH 1081   Paris CDG -> Algiers                    08:35 -> 10:20
  Tassili Airlines     SF 231    Paris ORY -> Oran                       11:10 -> 12:55
  Vueling              VY 8861   Paris CDG -> Algiers                    14:05 -> 15:50
note: Demo dataset served by the djef-flights MCP server — indicative prices, not a live quote.


## 2. Venues — live web search

The Venue Agent receives two parameters — `Destination` and `Guest_Count` — and
turns them into one targeted query for the **web search** tool (Tavily). Live
results with their links; without the key (or during an outage) the tool answers
honestly instead of failing.


In [6]:
res = json.loads(wp.web_search.invoke(
    {"query": "event venue hire Algiers capacity 300 guests price"}))

if "results" in res:
    for r in res["results"]:
        print(f"- {r['title'][:70]}\n    {r['url']}")
else:
    print(res["message"])


- Banquet Halls à Algiers
    https://3ersi.com/en/wedding-venue/search?wilaya=1&page=10
- Meetings & Events | Sofitel Algiers Hamma Gardens
    https://sofitel.accor.com/en/hotels/1540/meetings.html
- Event Space & Conference Rooms in Algiers - Algeria
    https://www.cvent.com/meeting-event-planning/algeria/algiers/venues
- Algiers
    https://www.magnolia-weddings.com/our-location/algiers


## 3. Playlist — SQL over a local database

**How the SQL database is created and connected** — four plain `sqlite3` steps
(this file *is* the database; SQLite needs no server):

1. **CREATE** — `CREATE TABLE IF NOT EXISTS tracks (...)` builds `data/playlist.db`
   the first time it runs;
2. **CONNECT** — `sqlite3.connect(wp.PLAYLIST_DB_PATH)` opens it;
3. **SEED** — a few demo rows for eight genres (`wp.PLAYLIST_SEED`);
4. **QUERY** — the Playlist Agent receives only `Genre` and answers with a
   `SELECT`; the tool allows `SELECT` only and refuses every write on purpose.

The service runs the same CREATE + seed for you on first use
(`ensure_playlist_db`), so the app always finds the same database.


In [7]:
import sqlite3

# 1. CREATE the table — the exact SQL that builds the database --------------
wp.PLAYLIST_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
con = sqlite3.connect(wp.PLAYLIST_DB_PATH)             # 2. CONNECT
con.execute(
    "CREATE TABLE IF NOT EXISTS tracks ("
    " id INTEGER PRIMARY KEY AUTOINCREMENT,"
    " title TEXT NOT NULL,"
    " artist TEXT NOT NULL,"
    " genre TEXT NOT NULL,"
    " duration_sec INTEGER NOT NULL)")
con.commit()

# 3. SEED it once with the service's demo rows
if con.execute("SELECT COUNT(*) FROM tracks").fetchone()[0] == 0:
    con.executemany(
        "INSERT INTO tracks (title, artist, genre, duration_sec)"
        " VALUES (?, ?, ?, ?)",
        wp.PLAYLIST_SEED)
    con.commit()

print("tables:", [r[0] for r in con.execute(
    "SELECT name FROM sqlite_master WHERE type='table'")])
schema = con.execute("SELECT sql FROM sqlite_master WHERE name='tracks'").fetchone()[0]
print(schema, "\n")

print("Rows per genre:")
for genre, n in con.execute(
        "SELECT genre, COUNT(*) AS n FROM tracks GROUP BY genre ORDER BY n DESC"):
    print(f"  {genre:9} {n}")

# 4. QUERY through the Playlist Agent's tool (the same SELECT it runs) ------
rows = json.loads(wp.run_sql.invoke(
    {"query": "SELECT title, artist, duration_sec FROM tracks "
              "WHERE genre LIKE '%chaabi%' LIMIT 4"}))
print(f"\nPlaylist for genre 'chaabi' ({rows['row_count']} rows):")
for r in rows["rows"]:
    minutes, seconds = divmod(r["duration_sec"], 60)
    print(f"  {r['title']} — {r['artist']}  ({minutes}:{seconds:02d})")

print("\nWrite attempt:", wp.run_sql.invoke({"query": "DELETE FROM tracks"}))
con.close()   # CLOSE it — the service opens its own connection the same way


tables: ['tracks', 'sqlite_sequence']
CREATE TABLE tracks ( id INTEGER PRIMARY KEY AUTOINCREMENT, title TEXT NOT NULL, artist TEXT NOT NULL, genre TEXT NOT NULL, duration_sec INTEGER NOT NULL) 

Rows per genre:
  rai       5
  pop       4
  kabyle    4
  chaabi    4
  rap       3
  melhoun   3
  andalous  3
  electro   2

Playlist for genre 'chaabi' (4 rows):
  Ya Rayah — Dahmane El Harrachi  (5:18)
  Nuit d'Alger — Orchestre El Djazaïr  (4:34)
  Zina — Group El Draâ  (4:05)
  Bab El Ghorba — Mahmoud Benali  (4:56)

Write attempt: {"error": "Only SELECT queries are allowed on this database.", "query": "DELETE FROM tracks"}


## 4. The Coordinator prompt

The coordinator's system prompt **is** the Coordinator Agent spec — role, the four
parameters, the three mappings, the workflow (extract → validate → dispatch →
synthesize) and the ✈️ 🏛️ 🎵 response format — plus one bridge section that binds
"dispatch" to the three real `ask_*` tools:

```python
print(wp.COORDINATOR_PROMPT)
```


In [8]:
print(wp.COORDINATOR_PROMPT)


# ROLE
You are the "Coordinator Agent," the central orchestrator for a multi-agent
travel and event planning system. Your job is to analyze user requests,
extract specific parameters, delegate tasks to specialized sub-agents, and
compile the final response.

# INPUT PARAMETERS TO EXTRACT
Whenever a user makes a request, you must extract the following variables:
- `Origin`: The starting location for flights.
- `Destination`: The location of the event/venue.
- `Guest_Count`: The number of people attending.
- `Genre`: The type of music or vibe for the playlist.

# SUB-AGENTS & TOOLS
You manage three specialized worker agents. You must route the extracted
parameters to them according to the strict mapping below:

1. **Flights Agent** (Tool: MCP Server)
   - **Parameter mapping:** `Origin`
   - **Task:** Retrieve flight information based on the origin.
2. **Venue Agent** (Tool: Web Search)
   - **Parameter mapping:** `Destination`, `Guest_Count`
   - **Task:** Search the web for suitable ve

## 5. The team — one coordinator, three specialists

The lesson `2_3` wiring, applied to a real system: every specialist is a normal
`create_agent` with its own real tool, and it hides behind an ordinary `@tool`
wrapper whose **signature carries exactly the mapped parameters**:

```python
@tool
def ask_venues(destination: str, guest_count: int) -> str:
    """Dispatch the Venue Agent (live web search)..."""
    return delegate(venue_agent,
                    f"Destination: {destination}\nGuest_Count: {guest_count}")
```

The coordinator sees three simple tools — it never knows it talks to agents.
Swapping a specialist later (another prompt, another model) would never touch
the coordinator.


In [9]:
agent = wp.build_event_agent(model=MODEL)

print("Coordinator (main) ->", wp.COORDINATOR_TOOLS)
for specialist, tools in wp.SPECIALISTS.items():
    print(f"  {specialist:9} specialist -> {tools}")


Coordinator (main) -> ['ask_flights', 'ask_venues', 'ask_playlist']
  flights   specialist -> ['search_flights (MCP server)']
  venue     specialist -> ['web_search (live web search)']
  playlist  specialist -> ['run_sql (SQL query)']


## 6. Test — extract, dispatch, synthesize

All four parameters in one request: the coordinator should call **the three
specialists in one turn** — `ask_flights(Origin)`, `ask_venues(Destination,
Guest_Count)`, `ask_playlist(Genre)` — then compile one answer shaped
✈️ Flights / 🏛️ Venue / 🎵 Playlist.


In [10]:
import time

from langchain.messages import HumanMessage

def live(run, tries=6, wait=20):
    # Free tier = 15 requests/min and one full coordinator run costs ~8 of
    # them, so a 429 here just waits and retries instead of failing the cell.
    for attempt in range(1, tries + 1):
        try:
            return run()
        except Exception as exc:
            if ("429" not in str(exc)) and ("RESOURCE_EXHAUSTED" not in str(exc)):
                raise
            if attempt == tries:
                raise
            print(f"  quota hit — waiting {wait}s ({attempt}/{tries})…")
            time.sleep(wait)
            wait = int(wait * 1.5)

question = (
    "Origin: Paris, Destination: Algiers, Guest_Count: 300, Genre: chaabi. "
    "Plan our event: flights from Paris, a venue in Algiers for 300 guests "
    "and a chaabi playlist."
)
response = live(lambda: agent.invoke({"messages": [HumanMessage(content=question)]}))
print(wp.last_text(response))


DJEF Wedding — a planner and her specialists. Here is the initial plan for your event in Algiers:

✈️ **Flights:**
*   **Air Algérie (AH 1081):** Paris CDG to Algiers, 08:35-10:20 (38,000 DA)
*   **Vueling (VY 8861):** Paris CDG to Algiers, 14:05-15:50 (34,900 DA)
*   *Note: These are demo flights provided by the flights MCP server. Prices are indicative and not live quotes.*

🏛️ **Venue:**
*   **Auberge des collines:** Capacity 300 guests, starting from 140,000 DA. [Website](https://3ersi.com/en/wedding-venue/search?wilaya=1&page=10)
*   **Sofitel Algiers Hamma Gardens:** The Salon Jardin d'Essai is perfectly suited for your 300 guests. [Website](https://sofitel.accor.com/en/hotels/1540/meetings.html)
*   **Hotel El-Aurassi:** Features extensive event spaces for larger gatherings. [Website](https://www.cvent.com/meeting-event-planning/algeria/algiers/venues)
*   *Note: These venue options were found via web search; please verify real-time availability and specific event requirements d

## 7. Validation — a missing parameter

Workflow step 2: with `Origin` and `Genre` missing, the coordinator must **ask**
for them instead of dispatching half a team.


In [11]:
response = live(lambda: agent.invoke({"messages": [HumanMessage(
    content="Find me a nice venue in Oran for the party.")]}))
print(wp.last_text(response))


I am the DJEF Wedding planner and my specialists. It sounds like you are planning a wonderful event in Oran!

To help me get everything organized for you, I need a few more details to work with my specialized agents:

1. **Which city are you flying from?** (I need this for the Flights Agent)
2. **How many guests are you expecting?** (I need this to help refine the venue search)
3. **What is the musical vibe/genre you'd like for the playlist?** (I need this for the Playlist Agent)

*Note: Please keep in mind that flight information is for demonstration, venue prices are indicative, and availability requires live research.* 

Once you provide these, I can get to work! Also, just a quick note to confirm—are you looking for venues in **Oran, Algeria**? (The search tools sometimes confuse this with locations in other countries like Scotland).


## 8. Trace — the same run, step by step

The printout above shows only the final answer. The trace shows the *decisions*:
which specialist the coordinator picked, with which arguments, node by node.

> Every `ask_*` call runs a whole specialist inside — one level deeper, that
> subagent has its own `model` and `tools` nodes (lesson `2_3`).


In [12]:
updates = live(lambda: list(agent.stream(
    {"messages": [HumanMessage(content=question)]}, stream_mode="updates")))

for update in updates:
    for node, payload in update.items():
        for msg in payload.get("messages", []):
            kind = type(msg).__name__.replace("Message", "")
            if getattr(msg, "tool_calls", None):
                call = msg.tool_calls[0]
                detail = f"-> {call['name']}({call['args']})"
            elif kind == "Tool":
                detail = f"{str(msg.content)[:70]}..."
            elif isinstance(msg.content, str):
                detail = msg.content[:70] + ("..." if len(msg.content) > 70 else "")
            elif msg.content:
                detail = msg.content[0].get("text", "")[:70] + "..."
            else:
                detail = ""
            print(f"[{node}] {kind}: {detail}")


  quota hit — waiting 20s (1/6)…
[model] AI: -> ask_flights({'origin': 'Paris'})
[tools] Tool: 1. Ya Rayah - Dahmane El Harrachi (318s)
2. Nuit d'Alger - Orchestre E...
[tools] Tool: Auberge des collines: Capacity 300 guests, starting from 140.000 DA. h...
[tools] Tool: Air Algérie · AH 1081 · Paris CDG -> Algiers · 08:35-10:20 · 38,000 DA...
[model] AI: DJEF Wedding — a planner and her specialists. I have organized the opt...


## 9. The Streamlit app

The app imports the same `wedding_planner_service` and keeps its own code
small — the UI is divided into parts, one concern per file:

- `design.py` — the look (page, header, footer)
- `components.py` — one widget / button per function
- `runner.py` — the functionality (keys, agent, context, saved runs)
- `app.py` — only the assembly

```bash
# double-click from the project folder:
run_app.cmd          # opens http://localhost:8511
```

The **wedding sidebar stays untouched** — Number of guests, Wilaya, Style,
Wedding date, Budget limit, Priorities, Saved runs — and *adds* two inputs
underneath: **Origin** and **Playlist genre** (Wilaya = `Destination`, guests =
`Guest_Count`). The byline sits on top with the 🌙 dark-mode switch. Type a
request → the coordinator dispatches the three specialists, and the ✈️ 🏛️ 🎵
answer is ready to **download as .md**.


## Summary of functions

| Function / class | Simple definition |
|---|---|
| `load_keys()` | Reads the Gemini key from the nearest `.env` (project or notebooks folder). |
| `flights_mcp_tools()` | Asks the local MCP server (`mcp_flights_server.py`) for its tools over stdio — each with a sync twin. |
| `@tool run_sql(query)` | SELECT-only SQL over the SQLite playlist database — inside the **playlist** specialist. |
| `@tool web_search(query)` | Live web search (Tavily) — inside **venue**; a friendly message without the key. |
| `ensure_playlist_db()` | Creates / seeds `data/playlist.db` (demo rows, 8 genres) when missing. |
| `COORDINATOR_PROMPT` | The Coordinator Agent ROLE spec + the bridge to the three `ask_*` tools. |
| `create_agent(model, tools, system_prompt)` | The factory — used for the coordinator and for every specialist. |
| `@tool ask_flights / ask_venues / ask_playlist` | The wrappers: ordinary tools carrying only the mapped parameters that run a subagent (lesson 2_3). |
| `agent.stream(..., stream_mode="updates")` | The trace: every node of the same run. |
| `last_text(response)` | Last message as plain text (Gemini content may be blocks). |


By
> Kassouar Fatima                   26-09-2026
